# Devoir 1 - Pipeline NLP pour structurer le Code de la Route marocain

**Objectif :** transformer le texte arabe brut du PDF de la loi 52.05 en un fichier `export_final.csv`, où chaque ligne décrit une règle, une obligation, une définition ou une sanction.

Le notebook applique deux approches complémentaires :

1. **Approche par règles** : normalisation arabe, segmentation par articles, Regex Unicode, extraction des montants, points, véhicules et mots-clés.
2. **Approche ML / modèle pré-entraîné** : catégorisation sémantique via AraBERT si disponible, avec fallback automatique TF-IDF + clustering pour que le notebook reste exécutable hors ligne.

> Placez le PDF `code de la route MA52_05.pdf` dans le même dossier que ce notebook, ou modifiez la variable `PDF_PATH` dans la cellule de configuration.

In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 69.2 MB/s eta 0:00:00


In [2]:
import sys
!{sys.executable} -m pip install pymupdf

# Dépendances minimales:
# pip install pymupdf pandas numpy scikit-learn
# Optionnel pour la version AraBERT:
# pip install torch transformers

from pathlib import Path
import re
import json
import unicodedata
import warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

try:
    import fitz  # PyMuPDF
except ImportError as exc:
    raise ImportError("Installez PyMuPDF: pip install pymupdf") from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)

In [3]:
# Configuration
PDF_CANDIDATES = [
    Path("/mnt/data/code de la route MA52_05.pdf"),
    Path("code de la route MA52_05.pdf"),
    Path("./data/code de la route MA52_05.pdf"),
]
PDF_PATH = next((p for p in PDF_CANDIDATES if p.exists()), None)
if PDF_PATH is None:
    raise FileNotFoundError("PDF introuvable. Placez 'code de la route MA52_05.pdf' dans le dossier du notebook.")

OUTPUT_CSV = Path("export_final.csv")
OUTPUT_JSONL = Path("export_final.jsonl")
print("PDF utilisé:", PDF_PATH)

PDF utilisé: code de la route MA52_05.pdf


In [4]:
# Utilitaires de normalisation arabe
ARABIC_TASHKEEL = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "\u0640"

ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
HAMZA_TRANSLATION = str.maketrans({
    "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا",
    "ؤ": "و", "ئ": "ي", "ء": "",
    "ى": "ي", "ة": "ه",
})


def normalize_digits(text: str) -> str:
    """Convertit les chiffres arabes / persans vers 0-9."""
    return text.translate(ARABIC_DIGITS)


def normalize_arabic(text: str) -> str:
    """Normalisation demandée: Hamza, Ta Marbuta, Tashkeel, Tatweel, espaces."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = normalize_digits(text)
    text = ARABIC_TASHKEEL.sub("", text)
    text = text.replace(TATWEEL, "")
    text = text.translate(HAMZA_TRANSLATION)
    # unifier بعض الاختلافات الناتجة عن extraction PDF
    text = text.replace("اال", "الا")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_text(text: str) -> str:
    text = normalize_digits(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    return text.strip()


def to_int(s: str) -> Optional[int]:
    if s is None:
        return None
    s = normalize_digits(str(s))
    s = re.sub(r"[^0-9]", "", s)
    return int(s) if s else None


def article_id_clean(raw: str) -> str:
    raw = normalize_digits(raw or "")
    raw = re.sub(r"\s+", "", raw)
    raw = raw.replace("–", "-").replace("—", "-")
    return raw

In [5]:
# Extraction du texte PDF avec conservation des pages

def extract_pdf_pages(pdf_path: Path) -> List[Dict]:
    doc = fitz.open(pdf_path)
    pages = []
    for idx, page in enumerate(doc, start=1):
        text = clean_text(page.get_text("text"))
        pages.append({"page": idx, "text": text})
    return pages

pages = extract_pdf_pages(PDF_PATH)
print(f"Nombre de pages extraites: {len(pages)}")
print(pages[0]["text"][:500])

Nombre de pages extraites: 126
تم إعداد هذه النسخة من أجل تسهيل 
مقروئية النص، وال يحتج إال بالنصوص 
في صيغتها املنشورة بالجريدة الرسمية
المتعلق52.05 القانون رقم
،بمدونة السير على الطرق
كما وقع تغييره وتتميمه
صيغة موطدة بتاريخ
2024 يوليو10


In [6]:
# Segmentation par article
# Le PDF extrait parfois "99 المادة" au lieu de "المادة 99"; on gère les deux formes.
ARTICLE_MARKER = re.compile(
    r"(?m)^\s*(?:(?:المادة|املادة)\s+(?P<id_after>[0-9]+(?:\s*[-–]\s*[0-9]+)?)|(?P<id_before>[0-9]+(?:\s*[-–]\s*[0-9]+)?)\s+(?:المادة|املادة))\s*$"
)
PAGE_MARKER = re.compile(r"@@PAGE:(\d+)@@")


def build_full_text_with_pages(pages: List[Dict]) -> str:
    parts = []
    for p in pages:
        parts.append(f"\n@@PAGE:{p['page']}@@\n")
        parts.append(p["text"])
    return "\n".join(parts)


def nearest_page_before(position: int, page_positions: List[Tuple[int, int]]) -> int:
    current = page_positions[0][1]
    for pos, page_num in page_positions:
        if pos <= position:
            current = page_num
        else:
            break
    return current


def segment_articles(pages: List[Dict]) -> pd.DataFrame:
    full = build_full_text_with_pages(pages)
    page_positions = [(m.start(), int(m.group(1))) for m in PAGE_MARKER.finditer(full)]
    markers = []
    for m in ARTICLE_MARKER.finditer(full):
        article_id = article_id_clean(m.group("id_after") or m.group("id_before"))
        if article_id:
            markers.append({"start": m.start(), "end": m.end(), "article_id": article_id})
    rows = []
    for i, m in enumerate(markers):
        next_start = markers[i + 1]["start"] if i + 1 < len(markers) else len(full)
        body = full[m["end"]:next_start]
        body = PAGE_MARKER.sub(" ", body)
        body = re.sub(r"\n?األمانة العامة للحكومة\n?", " ", body)
        body = re.sub(r"\n?الأمانة العامة للحكومة\n?", " ", body)
        body = clean_text(body)
        if len(body) < 20:
            continue
        rows.append({
            "article_id": m["article_id"],
            "source_page": nearest_page_before(m["start"], page_positions),
            "texte_original": body,
            "texte_normalise": normalize_arabic(body),
        })
    df = pd.DataFrame(rows).drop_duplicates(subset=["article_id", "texte_original"])
    return df.reset_index(drop=True)

articles_df = segment_articles(pages)
print("Articles segmentés:", len(articles_df))
articles_df.head(3)

Articles segmentés: 327


,article_id,source_page,texte_original,texte_normalise
0,1,2,ال يجوز ألي شخص أن يسوق مركبة ذات محرك أو مجموعة مركبات على الطريق العمومية\nما لم يكن حاصال على رخصة للسياقة سارية الصالحية ومسلمة من قبل اإلدارة، تناسب صنف\n.املركبة أو مجموع...,ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجموعه مركبات علي الطريق العموميه\nما لم يكن حاصال علي رخصه للسياقه ساريه الصالحيه ومسلمه من قبل الاداره، تناسب صنف\n.املركبه او مجموع...
1,2,2,: استثناء من أحكام املادة األولى أعاله\nيجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها- 1\nسنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السيا...,: استثنا من احكام املاده الاولي اعاله\nيجوز للمغاربه القاطنين بالخارج ان يسوقوا، داخل التراب الوطني، خالل مده اقصاها- 1\nسنه واحده ابتدا من اقامتهم باملغرب، بواسطه رخصه السياقه...
2,3,3,يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها\nفي املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تب...,يجب علي السايقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها\nفي املاده السابقه، ان يتقدموا المتحانات الحصول علي رخصه السياقه املغربيه، او ان يطلبوا تب...


In [7]:
# Dictionnaires métiers: catégories véhicules, mots-clés, rôles et types de sanction
VEHICLE_PATTERNS = {
    "poids_lourd": r"مركبات? .*?(?:يتجاوز وزنها|3500|3\.500|نقل البضائع|طن|الوزن الاجمالي|الوزن الإجمالي)",
    "vehicule_leger": r"السيارات المعدة لنقل الاشخاص|السيارات المعدة لنقل الأشخاص|صنف.*?ب|المركبة الخفيفة",
    "moto": r"دراجة نارية|الدراجات النارية|دراجة بمحرك|الدراجات.*?بمحرك|خوذة",
    "transport_collectif": r"النقل الجماعي|نقل الاشخاص|نقل الأشخاص|حافلات|حافلة|سيارات الاجرة|سيارات الأجرة",
    "taxi": r"سيارات الاجرة|سيارات الأجرة|الصنفين الاول والثاني|الصنفين الأول والثاني",
    "vehicule_agricole": r"مركبة فلاحية|المركبات الفلاحية|مركبة غابوية|المركبات الغابوية|أريبة|اريبة",
    "remorque": r"مقطورة|المقطورات|مجموعة مركبات",
    "tous_vehicules": r"كل مركبة|مركبة|المركبات",
}

KEYWORD_PATTERNS = {
    "vitesse": r"السرعة|كيلومترا في الساعة|رادار|جهاز قياس السرعة",
    "alcool_stupefiants": r"الكحول|المخدرة|مخدر|الرائز|اختبارات الكشف",
    "telephone": r"الهاتف|وظائف الهاتف",
    "ceinture": r"حزام السلامة",
    "casque": r"خوذة",
    "feu_rouge_stop": r"الضوء الاحمر|الضوء الأحمر|علامة قف|التوقف المفروض",
    "priorite": r"الأسبقية|الاولوية|الأولوية|حق الاسبقية",
    "depassement": r"التجاوز|تجاوز غير قانوني",
    "stationnement_arret": r"الوقوف|التوقف|التوقف العاجل|شريط التوقف",
    "autoroute": r"طريق سيار|الطريق السيار|طريق سريع|الطريق السريع",
    "eclairage": r"إنارة|انارة|تشوير|أضواء|اضواء",
    "controle_technique": r"المراقبة التقنية|شهادة المراقبة",
    "permis": r"رخصة السياقة|رخصة سياقة|رخصته للسياقة",
    "assurance": r"التأمين|تأمين المركبة|شهادة التأمين",
    "accident": r"حادثة سير|الحادثة|قتل غير عمدي|جروح غير عمدية",
    "charge_poids": r"الوزن|الحمولة|الشحن|إفراغ|افراغ|طن",
    "transport_professionnel": r"بطاقة سائق مهني|السائق المهني|السياقة بصفة مهنية|النقل العمومي",
    "enfant": r"طفل|عشر سنوات|المقاعد الامامية|المقاعد الأمامية",
}

ROLE_PATTERNS = {
    "definition": r"يراد في مفهوم|يقصد|تعريف|تعاريف|يعرف|تعني",
    "sanction": r"يعاقب|غرامة|حبس|توقيف|سحب|إلغاء|الغاء|خصم|نقط|المحجز|إيداع|ايداع",
    "obligation": r"يجب|لا يجوز|اليجوز|يلزم|يتعين|يحظر|يمنع|على كل|يتعرض كل|لا يمكن",
}

SANCTION_PATTERNS = {
    "amende": r"غرامة|درهم|درهما",
    "points": r"نقط|خصم",
    "prison": r"حبس|الحبس",
    "retrait_suspension_permis": r"توقيف رخصة|سحب رخصة|إلغاء رخصة|الغاء رخصة|الاحتفاظ برخصة",
    "fourriere": r"المحجز|إيداع|ايداع",
}


def find_labels(text: str, patterns: Dict[str, str]) -> List[str]:
    txt = normalize_arabic(text)
    labels = []
    for label, pat in patterns.items():
        if re.search(normalize_arabic(pat), txt):
            labels.append(label)
    return labels


def classify_role(text: str) -> str:
    labels = find_labels(text, ROLE_PATTERNS)
    if "sanction" in labels:
        return "sanction"
    if "obligation" in labels:
        return "obligation"
    if "definition" in labels:
        return "definition"
    return "autre"


def detect_vehicle_category(text: str) -> str:
    labels = find_labels(text, VEHICLE_PATTERNS)
    if not labels:
        return "non_precise"
    # tous_vehicules est utile mais moins spécifique: on le place à la fin
    labels = [x for x in labels if x != "tous_vehicules"] + (["tous_vehicules"] if "tous_vehicules" in labels else [])
    return ";".join(labels[:4])


def detect_keywords(text: str) -> List[str]:
    return find_labels(text, KEYWORD_PATTERNS)


def detect_sanction_types(text: str) -> List[str]:
    return find_labels(text, SANCTION_PATTERNS)

In [8]:
# Regex d'extraction: amendes, points, fourchettes, sanctions
AMOUNT_NEAR_DIRHAM = re.compile(r"(?:\(?\s*([0-9][0-9\.\s]{1,10})\s*\)?\s*(?:درهم|درهما))")
AMOUNT_RANGE_1 = re.compile(r"(?:من|بين)\s*\(?\s*([0-9][0-9\.\s]{1,10})\s*\)?\s*(?:درهم|درهما)?\s*(?:إلى|الى|و)\s*\(?\s*([0-9][0-9\.\s]{1,10})\s*\)?\s*(?:درهم|درهما)")
AMOUNT_RANGE_2 = re.compile(r"\(?\s*([0-9][0-9\.\s]{1,10})\s*\)?\s*(?:درهم|درهما)\s*(?:إلى|الى)\s*\(?\s*([0-9][0-9\.\s]{1,10})\s*\)?\s*(?:درهم|درهما)")
POINTS_PATTERNS = [
    re.compile(r"خصم\s+([0-9]{1,2})\s+نقط"),
    re.compile(r"([0-9]{1,2})\s+نقط"),
    re.compile(r"النقط\s+.*?([0-9]{1,2})"),
]


def parse_amount(raw: str) -> Optional[int]:
    if raw is None:
        return None
    # 1.400 أو 1 400 -> 1400
    raw = normalize_digits(str(raw))
    raw = raw.replace(".", "").replace(" ", "")
    raw = re.sub(r"[^0-9]", "", raw)
    return int(raw) if raw else None


def extract_amendes(text: str) -> Dict[str, Optional[int]]:
    norm = normalize_digits(text)
    ranges = []
    for pat in [AMOUNT_RANGE_1, AMOUNT_RANGE_2]:
        for m in pat.finditer(norm):
            a, b = parse_amount(m.group(1)), parse_amount(m.group(2))
            if a and b:
                ranges.append((min(a, b), max(a, b)))
    amounts = [parse_amount(m.group(1)) for m in AMOUNT_NEAR_DIRHAM.finditer(norm)]
    amounts = [x for x in amounts if x is not None and x >= 50]
    if ranges:
        return {
            "amende_min": min(x[0] for x in ranges),
            "amende_max": max(x[1] for x in ranges),
            "amende_fixe": None,
            "amendes_detectees": sorted(set(amounts)),
        }
    if amounts:
        unique = sorted(set(amounts))
        return {
            "amende_min": min(unique),
            "amende_max": max(unique),
            "amende_fixe": unique[0] if len(unique) == 1 else None,
            "amendes_detectees": unique,
        }
    return {"amende_min": None, "amende_max": None, "amende_fixe": None, "amendes_detectees": []}


def extract_points(text: str) -> Optional[int]:
    norm = normalize_digits(text)
    vals = []
    for pat in POINTS_PATTERNS:
        vals += [int(m.group(1)) for m in pat.finditer(norm) if 0 < int(m.group(1)) <= 16]
    return vals[0] if vals else None


def short_desc(text: str, max_len: int = 600) -> str:
    text = re.sub(r"\s+", " ", clean_text(text))
    # Supprimer les longs renvois de modification législative quand ils commencent l'article
    text = re.sub(r"^\([^)]{30,250}\)\s*", "", text)
    return text[:max_len].strip()

In [9]:
# Construction du DataFrame principal à partir des articles

def build_article_rows(articles_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in articles_df.iterrows():
        text = row["texte_original"]
        am = extract_amendes(text)
        points = extract_points(text)
        keywords = detect_keywords(text)
        role = classify_role(text)
        sanction_types = detect_sanction_types(text)
        rows.append({
            "regle_id": f"ART-{row['article_id']}",
            "article_id": row["article_id"],
            "source_page": int(row["source_page"]),
            "infraction_desc": short_desc(text),
            "categorie_vehicule": detect_vehicle_category(text),
            "amende_fixe": am["amende_fixe"],
            "amende_min": am["amende_min"],
            "amende_max": am["amende_max"],
            "amendes_detectees": ",".join(map(str, am["amendes_detectees"])),
            "points_retrait": points,
            "mots_cles": ",".join(keywords),
            "role_paragraphe": role,
            "type_sanction": ",".join(sanction_types),
            "approche_extraction": "regex_article",
            "texte_original": text,
            "texte_normalise": row["texte_normalise"],
        })
    return pd.DataFrame(rows)

article_rows = build_article_rows(articles_df)
print("Lignes article:", len(article_rows))
article_rows.head(5)

Lignes article: 327


,regle_id,article_id,source_page,infraction_desc,categorie_vehicule,amende_fixe,amende_min,amende_max,amendes_detectees,points_retrait,mots_cles,role_paragraphe,type_sanction,approche_extraction,texte_original,texte_normalise
0,ART-1,1,2,ال يجوز ألي شخص أن يسوق مركبة ذات محرك أو مجموعة مركبات على الطريق العمومية ما لم يكن حاصال على رخصة للسياقة سارية الصالحية ومسلمة من قبل اإلدارة، تناسب صنف .املركبة أو مجموعة ...,remorque;tous_vehicules,NaN,NaN,NaN,,NaN,,autre,,regex_article,ال يجوز ألي شخص أن يسوق مركبة ذات محرك أو مجموعة مركبات على الطريق العمومية\nما لم يكن حاصال على رخصة للسياقة سارية الصالحية ومسلمة من قبل اإلدارة، تناسب صنف\n.املركبة أو مجموع...,ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجموعه مركبات علي الطريق العموميه\nما لم يكن حاصال علي رخصه للسياقه ساريه الصالحيه ومسلمه من قبل الاداره، تناسب صنف\n.املركبه او مجموع...
1,ART-2,2,2,: استثناء من أحكام املادة األولى أعاله يجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها- 1 سنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السياقة...,non_precise,NaN,NaN,NaN,,NaN,"permis,charge_poids",autre,,regex_article,: استثناء من أحكام املادة األولى أعاله\nيجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها- 1\nسنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السيا...,: استثنا من احكام املاده الاولي اعاله\nيجوز للمغاربه القاطنين بالخارج ان يسوقوا، داخل التراب الوطني، خالل مده اقصاها- 1\nسنه واحده ابتدا من اقامتهم باملغرب، بواسطه رخصه السياقه...
2,ART-3,3,3,يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها في املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تبد...,non_precise,NaN,NaN,NaN,,NaN,"permis,charge_poids",obligation,,regex_article,يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها\nفي املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تب...,يجب علي السايقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها\nفي املاده السابقه، ان يتقدموا المتحانات الحصول علي رخصه السياقه املغربيه، او ان يطلبوا تب...
3,ART-4,4,3,في حالة السير الدولي ووفقا لالتفاقية الدولية للسير على الطرق، تسلم الهيئات املؤهلة لذلك من .قبل اإلدارة، رخصة دولية للسياقة موضوعة في دفتر خاص يجوز للسائقين من جنسية أجنبية، ال...,non_precise,NaN,NaN,NaN,,NaN,charge_poids,autre,,regex_article,في حالة السير الدولي ووفقا لالتفاقية الدولية للسير على الطرق، تسلم الهيئات املؤهلة لذلك من\n.قبل اإلدارة، رخصة دولية للسياقة موضوعة في دفتر خاص\nيجوز للسائقين من جنسية أجنبية، ...,في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهييات املوهله لذلك من\n.قبل الاداره، رخصه دوليه للسياقه موضوعه في دفتر خاص\nيجوز للسايقين من جنسيه اجنبيه، ...
4,ART-5,5,3,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5866 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يول...,vehicule_leger,NaN,NaN,NaN,,NaN,,autre,,regex_article,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم\n)5866 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يو...,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املاده الاويل من القانون رقم\n)5866 :)، ص2016 اغسطس11( 1437 ذي القعده7 بتاريخ6490 )، ج.ر عدد2016 يو...


In [10]:
# Extraction spéciale du tableau de l'article 99: infractions/points, pages 40-41 du PDF.
# Ce tableau est important car il donne les points à retirer par infraction.

def find_ordered_row_markers(sub: str, first: int = 1, last: int = 32) -> List[Tuple[int, int, int]]:
    """Trouve les numéros 01..32 du tableau, en évitant de confondre les points avec les IDs."""
    markers = []
    cursor = 0
    for num in range(first, last + 1):
        token = f"{num:02d}" if num < 10 else str(num)
        m = re.search(rf"(?m)^\s*{re.escape(token)}\b", sub[cursor:])
        if not m:
            # Pour quelques extractions, le numéro peut être au début d'une ligne suivi directement du texte.
            m = re.search(rf"(?m)^\s*{num}\b", sub[cursor:])
        if m:
            start = cursor + m.start()
            end = cursor + m.end()
            markers.append((start, end, num))
            cursor = end
    return markers


def extract_article99_points_table(pages: List[Dict]) -> pd.DataFrame:
    # Les pages PDF 40 et 41 contiennent le tableau (pagination interne 39 et 40).
    table_text = "\n".join(p["text"] for p in pages if p["page"] in [40, 41])
    # Supprimer les numéros de page et l'en-tête répétée qui peuvent couper la ligne 18.
    table_text = re.sub(r"(?m)^\s*[0-9]{1,3}\s*$\n\s*األمانة العامة للحكومة\s*$", "", table_text)
    table_text = clean_text(table_text)
    start = table_text.find("الجنح")
    end_candidates = [m.start() for m in re.finditer(r"(?m)^\s*100\s+(?:المادة|املادة)\s*$", table_text)]
    end = end_candidates[0] if end_candidates else len(table_text)
    sub = table_text[start:end]

    markers = find_ordered_row_markers(sub, 1, 32)
    rows = []
    for i, (s, e, num) in enumerate(markers):
        next_s = markers[i + 1][0] if i + 1 < len(markers) else len(sub)
        seg = clean_text(sub[e:next_s])
        # point = dernier petit nombre isolé dans le segment; il peut être collé au texte
        # ou suivi par l'en-tête de la page suivante.
        norm_seg = normalize_digits(seg)
        point_matches = [m for m in re.finditer(r"(?<![0-9])([0-9]{1,2})(?![0-9])", norm_seg) if 1 <= int(m.group(1)) <= 16]
        if not point_matches:
            continue
        point_m = point_matches[-1]
        points = int(point_m.group(1))
        desc = norm_seg[:point_m.start()].strip()
        desc = re.sub(r"^(?:الرقم|الترتيبي|الجنحة|المخالفات|النقط|الواجب|خصمها)\s+", "", desc).strip(" .،-\n")
        if len(desc) < 8:
            continue
        rows.append({
            "regle_id": f"ART-99-{num:02d}",
            "article_id": "99",
            "source_page": 40 if num <= 18 else 41,
            "infraction_desc": short_desc(desc, max_len=500),
            "categorie_vehicule": detect_vehicle_category(desc),
            "amende_fixe": None,
            "amende_min": None,
            "amende_max": None,
            "amendes_detectees": "",
            "points_retrait": points,
            "mots_cles": ",".join(detect_keywords(desc)),
            "role_paragraphe": "sanction",
            "type_sanction": "points",
            "approche_extraction": "regex_table_article_99",
            "texte_original": desc,
            "texte_normalise": normalize_arabic(desc),
        })
    df = pd.DataFrame(rows).drop_duplicates(subset=["regle_id"])
    return df

points99_df = extract_article99_points_table(pages)
print("Lignes extraites du tableau de l'article 99:", len(points99_df))
points99_df[["regle_id", "infraction_desc", "points_retrait", "mots_cles"]].head(10)

Lignes extraites du tableau de l'article 99: 32


,regle_id,infraction_desc,points_retrait,mots_cles
0,ART-99-01,القتل غير العمدي مع ظروف التشديد، إثر حادثة سير (ما لم يتقرر إلغاء .)رخصة السياقة,14,"permis,accident"
1,ART-99-02,القتل غير العمدي بدون ظروف التشديد، إثر حادثة سير,6,accident
2,ART-99-03,الجروح غير العمدية املؤدية إلى عاهة دائمة مع ظروف التشديد، إثر .)حادثة سير ( ما لم يتقرر إلغاء رخصة السياقة,10,"permis,accident"
3,ART-99-04,الجروح غير العمدية املؤدية إلى عاهة دائمة بدون ظروف التشديد، إثر .حادثة سير,4,accident
4,ART-99-05,الجروح غير العمدية مع ظروف التشديد، إثر حادثة سير,6,accident
5,ART-99-06,الجروح غير العمدية بدون ظروف التشديد، إثر حادثة سير,3,accident
6,ART-99-07,سياقة مركبة تحت تأثير الكحول أو تحت تأثير املواد املخدرة - أدناه أو للتحققات207 رفض الخضوع للرائز املشار إليه في املادة- . أدناه213 و208 أو الختبارات الكشف املنصوص عليها في امل...,6,alcool_stupefiants
7,ART-99-08,سياقة مركبة تحت تأثير األدوية التي تحظر السياقة بعد تناولها,2,
8,ART-99-09,محاولة التملص من املسؤولية بعدم التوقف، بعد ارتكاب حادثة سير أو التسبب فيها، أو بالفرار أو بتغيير حالة مكان الحادثة أو بأية وسيلة .أخرى,6,"stationnement_arret,accident"
9,ART-99-10,سياقة مركبة، تتطلب سياقتها الحصول على رخصة السياقة، بالرغم .من توقيف إداري أو قضائي لرخصة السياقة,4,permis


In [11]:
# Fusion: conserver les lignes article + les lignes détaillées de l'article 99.
# Pour l'export final, on privilégie les lignes utiles: sanctions/obligations/définitions ou lignes contenant des amendes/points/mots-clés.

df = pd.concat([article_rows, points99_df], ignore_index=True)

useful_mask = (
    df["role_paragraphe"].isin(["sanction", "obligation", "definition"])
    | df["amende_min"].notna()
    | df["points_retrait"].notna()
    | df["mots_cles"].astype(str).str.len().gt(0)
)
export_df = df.loc[useful_mask].copy()
export_df = export_df.drop_duplicates(subset=["regle_id", "infraction_desc"])

# Ordonner les colonnes demandées + colonnes supplémentaires significatives.
ordered_cols = [
    "regle_id", "article_id", "source_page", "infraction_desc", "categorie_vehicule",
    "amende_fixe", "amende_min", "amende_max", "amendes_detectees",
    "points_retrait", "mots_cles", "role_paragraphe", "type_sanction",
    "approche_extraction", "texte_original", "texte_normalise",
]
export_df = export_df[ordered_cols]

print("Lignes finales avant ML:", len(export_df))
export_df.head(8)

Lignes finales avant ML: 313


,regle_id,article_id,source_page,infraction_desc,categorie_vehicule,amende_fixe,amende_min,amende_max,amendes_detectees,points_retrait,mots_cles,role_paragraphe,type_sanction,approche_extraction,texte_original,texte_normalise
1,ART-2,2,2,: استثناء من أحكام املادة األولى أعاله يجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها- 1 سنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السياقة...,non_precise,NaN,NaN,NaN,,NaN,"permis,charge_poids",autre,,regex_article,: استثناء من أحكام املادة األولى أعاله\nيجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها- 1\nسنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السيا...,: استثنا من احكام املاده الاولي اعاله\nيجوز للمغاربه القاطنين بالخارج ان يسوقوا، داخل التراب الوطني، خالل مده اقصاها- 1\nسنه واحده ابتدا من اقامتهم باملغرب، بواسطه رخصه السياقه...
2,ART-3,3,3,يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها في املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تبد...,non_precise,NaN,NaN,NaN,,NaN,"permis,charge_poids",obligation,,regex_article,يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها\nفي املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تب...,يجب علي السايقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها\nفي املاده السابقه، ان يتقدموا المتحانات الحصول علي رخصه السياقه املغربيه، او ان يطلبوا تب...
3,ART-4,4,3,في حالة السير الدولي ووفقا لالتفاقية الدولية للسير على الطرق، تسلم الهيئات املؤهلة لذلك من .قبل اإلدارة، رخصة دولية للسياقة موضوعة في دفتر خاص يجوز للسائقين من جنسية أجنبية، ال...,non_precise,NaN,NaN,NaN,,NaN,charge_poids,autre,,regex_article,في حالة السير الدولي ووفقا لالتفاقية الدولية للسير على الطرق، تسلم الهيئات املؤهلة لذلك من\n.قبل اإلدارة، رخصة دولية للسياقة موضوعة في دفتر خاص\nيجوز للسائقين من جنسية أجنبية، ...,في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهييات املوهله لذلك من\n.قبل الاداره، رخصه دوليه للسياقه موضوعه في دفتر خاص\nيجوز للسايقين من جنسيه اجنبيه، ...
5,ART-6,6,4,ال يجوز ألي كان سياقة مركبة فالحية ذات محرك أو مركبة غابوية ذات محرك أو أريبة لألشغال العمومية أو أريبة خاصة ذات محرك، على الطريق العمومية، مالم يكن حاصال على رخصة للسياقة .مسل...,vehicule_agricole;tous_vehicules,NaN,NaN,NaN,,NaN,permis,autre,,regex_article,ال يجوز ألي كان سياقة مركبة فالحية ذات محرك أو مركبة غابوية ذات محرك أو أريبة لألشغال\nالعمومية أو أريبة خاصة ذات محرك، على الطريق العمومية، مالم يكن حاصال على رخصة للسياقة\n.م...,ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او مركبه غابويه ذات محرك او اريبه لالشغال\nالعموميه او اريبه خاصه ذات محرك، علي الطريق العموميه، مالم يكن حاصال علي رخصه للسياقه\n.م...
6,ART-7,7,4,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5866 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يول...,vehicule_leger;moto;transport_collectif;remorque,NaN,NaN,NaN,,NaN,"permis,charge_poids",sanction,,regex_article,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم\n)5866 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يو...,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املاده الاويل من القانون رقم\n)5866 :)، ص2016 اغسطس11( 1437 ذي القعده7 بتاريخ6490 )، ج.ر عدد2016 يو...
7,ART-8,8,6,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5866 :)، ص2016 أغسطس11(1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يولي...,vehicule_leger;moto,NaN,NaN,NaN,,NaN,permis,autre,,regex_article,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم\n)5866 :)، ص2016 أغسطس11(1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يول...,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املاده الاويل من القانون رقم\n)5866 :)، ص2016 اغسطس11(1437 ذي القعده7 بتاريخ6490 )، ج.ر عدد2016 يو

In [12]:
# Approche ML 1: vectorisation TF-IDF + clustering non supervisé.
# Cette partie fonctionne hors ligne et regroupe les règles similaires (vitesse, stationnement, documents, etc.).

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

texts = export_df["texte_normalise"].fillna("").astype(str).tolist()

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    token_pattern=r"(?u)\b\w+\b",
)
X = vectorizer.fit_transform(texts)

n_clusters = min(12, max(2, len(export_df) // 25))
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
export_df["cluster_tfidf"] = kmeans.fit_predict(X)

# Catégorisation par similarité à des prototypes métiers en arabe.
CATEGORY_PROTOTYPES = {
    "vitesse": "تجاوز السرعة القصوى المسموح بها رادار كيلومتر في الساعة",
    "alcool_stupefiants": "السياقة تحت تأثير الكحول أو المواد المخدرة اختبارات الكشف",
    "documents_permis": "رخصة السياقة شهادة التسجيل شهادة التأمين الوثائق",
    "stationnement_arret": "الوقوف التوقف شريط التوقف العاجل المحجز",
    "securite_passagers": "حزام السلامة خوذة طفل المقاعد الأمامية الركاب",
    "priorite_signalisation": "علامة قف الضوء الأحمر الأسبقية التشوير الاتجاه الممنوع",
    "depassement_sens": "التجاوز غير القانوني الاتجاه المعاكس نصف دورة الطريق السيار",
    "controle_technique": "المراقبة التقنية أجهزة السلامة العيوب التقنية",
    "poids_charge": "الوزن الإجمالي الحمولة الشحن الطن نقل البضائع",
    "transport_professionnel": "بطاقة السائق المهني النقل الجماعي النقل العمومي للأشخاص",
    "accident_responsabilite": "حادثة سير قتل غير عمدي جروح الفرار المسؤولية",
}

prototype_texts = [normalize_arabic(v) for v in CATEGORY_PROTOTYPES.values()]
proto_X = vectorizer.transform(prototype_texts)
sim = cosine_similarity(X, proto_X)
best_idx = sim.argmax(axis=1)
export_df["categorie_ml_tfidf"] = [list(CATEGORY_PROTOTYPES.keys())[i] for i in best_idx]
export_df["score_ml_tfidf"] = sim.max(axis=1).round(3)

export_df[["regle_id", "infraction_desc", "cluster_tfidf", "categorie_ml_tfidf", "score_ml_tfidf"]].head(10)

,regle_id,infraction_desc,cluster_tfidf,categorie_ml_tfidf,score_ml_tfidf
1,ART-2,: استثناء من أحكام املادة األولى أعاله يجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها- 1 سنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السياقة...,7,documents_permis,0.028
2,ART-3,يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها في املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تبد...,0,documents_permis,0.042
3,ART-4,في حالة السير الدولي ووفقا لالتفاقية الدولية للسير على الطرق، تسلم الهيئات املؤهلة لذلك من .قبل اإلدارة، رخصة دولية للسياقة موضوعة في دفتر خاص يجوز للسائقين من جنسية أجنبية، ال...,2,documents_permis,0.017
5,ART-6,ال يجوز ألي كان سياقة مركبة فالحية ذات محرك أو مركبة غابوية ذات محرك أو أريبة لألشغال العمومية أو أريبة خاصة ذات محرك، على الطريق العمومية، مالم يكن حاصال على رخصة للسياقة .مسل...,5,documents_permis,0.028
6,ART-7,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5866 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يول...,11,poids_charge,0.113
7,ART-8,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5866 :)، ص2016 أغسطس11(1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يولي...,7,documents_permis,0.067
8,ART-9,يجب اإلدالء برخصة السياقة أو بالوثيقة التي تحل محلها إلى األعوان املكلفين بمراقبة تطبيق .أحكام هذا القانون والنصوص الصادرة لتطبيقه، كلما طلبوا ذلك 6 الباب الثالث شروط الحصول عل...,7,documents_permis,0.035
9,ART-10,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5866 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يول...,5,controle_technique,0.102
10,ART-11,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5867 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يول...,1,documents_permis,0.053
11,ART-12,،يخضع وجوبا كل مترشح الختبارات امتحان الحصول على رخصة السياقة لفحص طبي مسبق الغاية منه التأكد من أن قدراته البدنية والعقلية تمكنه من سياقة مركبة على الطريق العمومية دون خطر وخا...,7,documents_permis,0.042


In [13]:
# Approche ML 2 optionnelle: AraBERT pré-entraîné.
# Elle est activée si transformers + torch + modèle disponible sont installés.
# Sinon, le notebook conserve les catégories TF-IDF pour rester reproductible.

USE_ARABERT = True  # mettre False si vous ne voulez pas charger le modèle


def try_arabert_classification(texts: List[str], prototypes: Dict[str, str]):
    if not USE_ARABERT:
        return None, "AraBERT désactivé par configuration."
    try:
        import torch
        from transformers import AutoTokenizer, AutoModel
        model_name = "aubmindlab/bert-base-arabertv02"
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
        model.eval()

        def embed(batch_texts, batch_size=16):
            vectors = []
            with torch.no_grad():
                for i in range(0, len(batch_texts), batch_size):
                    batch = batch_texts[i:i+batch_size]
                    enc = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
                    out = model(**enc).last_hidden_state
                    mask = enc["attention_mask"].unsqueeze(-1)
                    pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                    vectors.append(pooled.cpu().numpy())
            return np.vstack(vectors)

        text_vecs = embed(texts)
        proto_names = list(prototypes.keys())
        proto_vecs = embed([normalize_arabic(v) for v in prototypes.values()])
        sims = cosine_similarity(text_vecs, proto_vecs)
        idx = sims.argmax(axis=1)
        return pd.DataFrame({
            "categorie_ml_arabert": [proto_names[i] for i in idx],
            "score_ml_arabert": sims.max(axis=1).round(3),
            "modele_preentraine": model_name,
        }), None
    except Exception as exc:
        return None, str(exc)

arabert_df, arabert_error = try_arabert_classification(texts, CATEGORY_PROTOTYPES)
if arabert_df is not None:
    export_df = pd.concat([export_df.reset_index(drop=True), arabert_df.reset_index(drop=True)], axis=1)
    export_df["categorie_ml_finale"] = export_df["categorie_ml_arabert"]
    print("AraBERT utilisé avec succès.")
else:
    export_df["categorie_ml_arabert"] = ""
    export_df["score_ml_arabert"] = np.nan
    export_df["modele_preentraine"] = "TF-IDF fallback (AraBERT non chargé)"
    export_df["categorie_ml_finale"] = export_df["categorie_ml_tfidf"]
    print("AraBERT non utilisé. Fallback TF-IDF activé.")
    print("Raison:", arabert_error)

export_df[["regle_id", "categorie_ml_finale", "modele_preentraine"]].head()

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AraBERT utilisé avec succès.


,regle_id,categorie_ml_finale,modele_preentraine
0,ART-2,transport_professionnel,aubmindlab/bert-base-arabertv02
1,ART-3,vitesse,aubmindlab/bert-base-arabertv02
2,ART-4,documents_permis,aubmindlab/bert-base-arabertv02
3,ART-6,documents_permis,aubmindlab/bert-base-arabertv02
4,ART-7,vitesse,aubmindlab/bert-base-arabertv02


In [14]:
# Nettoyage final et export CSV / JSONL

# Conversion de types pour un CSV propre
for col in ["amende_fixe", "amende_min", "amende_max", "points_retrait"]:
    export_df[col] = pd.to_numeric(export_df[col], errors="coerce").astype("Int64")

final_cols = [
    "regle_id", "article_id", "source_page", "infraction_desc", "categorie_vehicule",
    "amende_fixe", "amende_min", "amende_max", "amendes_detectees", "points_retrait",
    "mots_cles", "role_paragraphe", "type_sanction",
    "categorie_ml_finale", "cluster_tfidf", "score_ml_tfidf",
    "categorie_ml_arabert", "score_ml_arabert", "modele_preentraine",
    "approche_extraction", "texte_original", "texte_normalise",
]
export_df = export_df[final_cols].sort_values(["source_page", "article_id", "regle_id"]).reset_index(drop=True)

export_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
export_df.to_json(OUTPUT_JSONL, orient="records", force_ascii=False, lines=True)

print("CSV créé:", OUTPUT_CSV.resolve())
print("JSONL créé:", OUTPUT_JSONL.resolve())
print("Nombre de lignes:", len(export_df))
print("Colonnes:", list(export_df.columns))
export_df.head(12)

CSV créé: /content/export_final.csv
JSONL créé: /content/export_final.jsonl
Nombre de lignes: 313
Colonnes: ['regle_id', 'article_id', 'source_page', 'infraction_desc', 'categorie_vehicule', 'amende_fixe', 'amende_min', 'amende_max', 'amendes_detectees', 'points_retrait', 'mots_cles', 'role_paragraphe', 'type_sanction', 'categorie_ml_finale', 'cluster_tfidf', 'score_ml_tfidf', 'categorie_ml_arabert', 'score_ml_arabert', 'modele_preentraine', 'approche_extraction', 'texte_original', 'texte_normalise']


,regle_id,article_id,source_page,infraction_desc,categorie_vehicule,amende_fixe,amende_min,amende_max,amendes_detectees,points_retrait,...,type_sanction,categorie_ml_finale,cluster_tfidf,score_ml_tfidf,categorie_ml_arabert,score_ml_arabert,modele_preentraine,approche_extraction,texte_original,texte_normalise
0,ART-2,2,2,: استثناء من أحكام املادة األولى أعاله يجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها- 1 سنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السياقة...,non_precise,<NA>,<NA>,<NA>,,<NA>,...,,transport_professionnel,7,0.028,transport_professionnel,0.637,aubmindlab/bert-base-arabertv02,regex_article,: استثناء من أحكام املادة األولى أعاله\nيجوز للمغاربة القاطنين بالخارج أن يسوقوا، داخل التراب الوطني، خالل مدة أقصاها- 1\nسنة واحدة ابتداء من إقامتهم باملغرب، بواسطة رخصة السيا...,: استثنا من احكام املاده الاولي اعاله\nيجوز للمغاربه القاطنين بالخارج ان يسوقوا، داخل التراب الوطني، خالل مده اقصاها- 1\nسنه واحده ابتدا من اقامتهم باملغرب، بواسطه رخصه السياقه...
1,ART-3,3,3,يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها في املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تبد...,non_precise,<NA>,<NA>,<NA>,,<NA>,...,,vitesse,0,0.042,vitesse,0.641,aubmindlab/bert-base-arabertv02,regex_article,يجب على السائقين الحاصلين على رخصة سياقة مسلمة بالخارج، بعد انصرام املدة املشار إليها\nفي املادة السابقة، أن يتقدموا المتحانات الحصول على رخصة السياقة املغربية، أو أن يطلبوا تب...,يجب علي السايقين الحاصلين علي رخصه سياقه مسلمه بالخارج، بعد انصرام املده املشار اليها\nفي املاده السابقه، ان يتقدموا المتحانات الحصول علي رخصه السياقه املغربيه، او ان يطلبوا تب...
2,ART-4,4,3,في حالة السير الدولي ووفقا لالتفاقية الدولية للسير على الطرق، تسلم الهيئات املؤهلة لذلك من .قبل اإلدارة، رخصة دولية للسياقة موضوعة في دفتر خاص يجوز للسائقين من جنسية أجنبية، ال...,non_precise,<NA>,<NA>,<NA>,,<NA>,...,,documents_permis,2,0.017,documents_permis,0.700,aubmindlab/bert-base-arabertv02,regex_article,في حالة السير الدولي ووفقا لالتفاقية الدولية للسير على الطرق، تسلم الهيئات املؤهلة لذلك من\n.قبل اإلدارة، رخصة دولية للسياقة موضوعة في دفتر خاص\nيجوز للسائقين من جنسية أجنبية، ...,في حاله السير الدولي ووفقا لالتفاقيه الدوليه للسير علي الطرق، تسلم الهييات املوهله لذلك من\n.قبل الاداره، رخصه دوليه للسياقه موضوعه في دفتر خاص\nيجوز للسايقين من جنسيه اجنبيه، ...
3,ART-6,6,4,ال يجوز ألي كان سياقة مركبة فالحية ذات محرك أو مركبة غابوية ذات محرك أو أريبة لألشغال العمومية أو أريبة خاصة ذات محرك، على الطريق العمومية، مالم يكن حاصال على رخصة للسياقة .مسل...,vehicule_agricole;tous_vehicules,<NA>,<NA>,<NA>,,<NA>,...,,documents_permis,5,0.028,documents_permis,0.668,aubmindlab/bert-base-arabertv02,regex_article,ال يجوز ألي كان سياقة مركبة فالحية ذات محرك أو مركبة غابوية ذات محرك أو أريبة لألشغال\nالعمومية أو أريبة خاصة ذات محرك، على الطريق العمومية، مالم يكن حاصال على رخصة للسياقة\n.م...,ال يجوز الي كان سياقه مركبه فالحيه ذات محرك او مركبه غابويه ذات محرك او اريبه لالشغال\nالعموميه او اريبه خاصه ذات محرك، علي الطريق العموميه، مالم يكن حاصال علي رخصه للسياقه\n.م...
4,ART-7,7,4,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5866 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يول...,vehicule_leger;moto;transport_collectif;remorque,<NA>,<NA>,<NA>,,<NA>,...,,vitesse,11,0.113,vitesse,0.599,aubmindlab/bert-base-arabertv02,regex_article,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم\n)5866 :)، ص2016 أغسطس11( 1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يو...,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املاده الاويل من القانون رقم\n)5866 :)، ص2016 اغسطس11( 1437 ذي القعده7 بتاريخ6490 )، ج.ر عدد2016 يو...
5,ART-8,8,6,من13 بتاريخ1.16.106 الصادر بتنفيذه الظهري الرشيف رقم116.14 (غريت ومتمت مبوجب املادة األوىل من القانون رقم )5866 :)، ص2016 أغسطس11(1437 ذي القعدة7 بتاريخ6490 )، ج.ر عدد2016 يولي...,vehicule_leger;moto,<NA>,<NA>,<NA>,,<NA>,

## Résultat attendu

À la fin de l'exécution, le notebook génère :

- `export_final.csv` : fichier demandé dans le devoir.
- `export_final.jsonl` : version alternative utile pour l'audit ou l'indexation NLP.

Les colonnes principales demandées sont présentes : `article_id`, `infraction_desc`, `categorie_vehicule`, `amende_fixe`, `points_retrait`, `mots_cles`.

Colonnes supplémentaires ajoutées : `regle_id`, `source_page`, `amende_min`, `amende_max`, `role_paragraphe`, `type_sanction`, `categorie_ml_finale`, `cluster_tfidf`, `texte_original`, `texte_normalise`.